# 03 — Air Quality Interpolation (PM2.5)

Three-step workflow:

| Step | Parameter | This notebook |
|------|-----------|---------------|
| 1 | `data=` | OpenAQ PM2.5 stations (or offline synthetic) |
| 2 | `boundary=` | Named place or 4-corner bbox |
| 3 | `method=` | IDW · Kriging · Random Forest · GBM · Regression Kriging |

**Offline cells** run without any network access.  
**🌐 Network cells** fetch live OpenAQ data.  
**🛰 GEE cells** validate against Sentinel-5P aerosol index (requires `earthengine authenticate`).

In [ ]:
%pip install -q "geointerpo[full]" geemap localtileserver

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import pandas as pd
from geointerpo import Pipeline

## Configuration

In [ ]:
BOUNDARY  = (68.0, 20.0, 90.0, 35.0)  # India / Pakistan — dense OpenAQ coverage
DATE      = '2024-01-15'
RESOLUTION = 0.5  # degrees (~50 km)

---
## Part A — Offline demo (no network needed)

In [ ]:
result = Pipeline(
    data='sample',                 # Step 1 — built-in synthetic PM2.5 data
    variable='air_quality',
    boundary=BOUNDARY,             # Step 2 — four corners
    method=['idw', 'kriging', 'rf', 'gbm'],  # Step 3
    method_params={
        'idw':    {'power': 2},
        'kriging': {'variogram_model': 'exponential'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

In [ ]:
print('Cross-validation metrics:')
result.metrics_table()

In [ ]:
result.plot(cmap='YlOrRd')
plt.suptitle('PM2.5 — synthetic data (offline demo)', y=1.02)
plt.show()

### Station observations

In [ ]:
ax = result.stations.plot(
    column='value', cmap='YlOrRd', legend=True,
    figsize=(9, 6), markersize=40, edgecolor='k', linewidth=0.4
)
ax.set_title('PM2.5 station observations (synthetic, µg/m³)')
plt.tight_layout()

### City boundary clipping

Restrict the output to a specific city boundary — bbox is derived automatically.

In [ ]:
# Offline: use a shapely bbox polygon as the boundary
from shapely.geometry import box
delhi_box = box(76.8, 28.4, 77.4, 28.9)

result_delhi = Pipeline(
    data='sample',
    variable='air_quality',
    boundary=delhi_box,            # Shapely polygon — clipped automatically
    method=['idw', 'kriging'],
    resolution=0.1,
    cv_folds=3,
).run()

result_delhi.plot(cmap='YlOrRd')
plt.suptitle('PM2.5 — Delhi region (clipped to boundary)', y=1.02)
plt.show()

---
## Part B — 🌐 Live OpenAQ data

OpenAQ free tier — no API key required (~1000 requests/day without key).

In [ ]:
result_live = Pipeline(
    data='openaq',                 # Step 1 — live OpenAQ API
    variable='pm25',
    date=DATE,
    boundary=BOUNDARY,             # Step 2
    method=['idw', 'kriging', 'rf', 'gbm'],  # Step 3
    method_params={
        'kriging': {'variogram_model': 'exponential'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

print(f'Loaded {len(result_live.stations)} OpenAQ stations')

In [ ]:
result_live.metrics_table()

In [ ]:
result_live.plot(cmap='YlOrRd')
plt.suptitle(f'PM2.5 — {DATE}  (OpenAQ)', y=1.02)
plt.show()

### Regression Kriging with elevation covariate

Uses SRTM elevation as a covariate — topography can influence PM2.5 dispersion.

In [ ]:
result_rk = Pipeline(
    data='sample',
    variable='air_quality',
    boundary=BOUNDARY,
    method=['kriging', 'rk'],
    method_params={'kriging': {'variogram_model': 'exponential'}},
    include_dem=True,
    dem_source='synthetic',        # use 'gee' or 'srtm' with network
    resolution=RESOLUTION,
    cv_folds=5,
).run()

print('Ordinary Kriging vs Regression Kriging (with DEM):')
result_rk.metrics_table()

### Interactive map

In [ ]:
import geemap
import tempfile

da = result.grid
with tempfile.NamedTemporaryFile(suffix=".tif", delete=False) as f:
    tmp = f.name
da.rio.set_spatial_dims(x_dim="lon", y_dim="lat").rio.write_crs("EPSG:4326").rio.to_raster(tmp)

try:
    m = geemap.Map(center=[float(da.lat.mean()), float(da.lon.mean())], zoom=6, ee_initialize=False)
    m.add_raster(tmp, colormap="ylorrd", layer_name="PM2.5")
    m.add_layer_manager()
    display(m)
except Exception as e:
    print(f"Interactive map unavailable ({e}); falling back to static plot")
    result.plot(cmap='YlOrRd')
    plt.show()

---
## Part C — 🛰 GEE validation against Sentinel-5P

Validates the interpolated surface against Sentinel-5P aerosol index.

**Setup (run once):**
```bash
earthengine authenticate
```

In [ ]:
%pip install -q "geointerpo[gee]" geemap localtileserver

In [ ]:
import ee
import geemap

GEE_PROJECT = 'ee-alberta'

geemap.ee_initialize(project=GEE_PROJECT)
print(f'✓ GEE initialized with project: {GEE_PROJECT}')

In [ ]:
result_gee = Pipeline(
    data='openaq',
    variable='pm25',
    date=DATE,
    boundary=BOUNDARY,
    method=['idw', 'kriging', 'rf'],
    method_params={'kriging': {'variogram_model': 'exponential'}},
    resolution=RESOLUTION,
    validate_with_gee=True,
).run()

print('GEE validation (vs Sentinel-5P AAI):')
for k, v in result_gee.gee_metrics.items():
    if k != 'diff_map':
        print(f'  {k}: {v:.3f}')

In [ ]:
import tempfile
from geointerpo import viz

fig = viz.plot_diff(
    result_gee.gee_reference,
    result_gee.grids['kriging'],
    title='Kriging − Sentinel-5P AAI'
)
plt.show()

m = geemap.Map(center=[float(result_gee.grid.lat.mean()), float(result_gee.grid.lon.mean())], zoom=5)
for name, da in [('Kriging', result_gee.grids['kriging']), ('Sentinel-5P', result_gee.gee_reference)]:
    with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as f:
        tmp = f.name
    da.rio.set_spatial_dims(x_dim='lon', y_dim='lat').rio.write_crs('EPSG:4326').rio.to_raster(tmp)
    m.add_raster(tmp, colormap='ylorrd', layer_name=name)

m.add_layer_manager()
m

### All-method GEE comparison

In [ ]:
from geointerpo.validation import compute_metrics
import numpy as np

rows = []
ref = result_gee.gee_reference
for method, grid in result_gee.grids.items():
    try:
        m = compute_metrics(ref.values.ravel(), grid.values.ravel())
        rows.append({'method': method, **{k: round(v, 3) for k, v in m.items()}})
    except Exception:
        pass

pd.DataFrame(rows).set_index('method')

---
## Save outputs

In [ ]:
result.save('outputs/air_quality', geotiff=True, plot=True)
print('Saved to outputs/air_quality/')